In [1]:
import pandas as pd
import pathlib
import ollama
import sys

In [2]:
DATABASE_GOOD = "sqlite:///../data/track2_good.db"
DATABASE_BAD = "sqlite:///../data/track2_bad.db"
OUTPUT = '../data/output.txt'
PROMPTS = '../prompts/prompts.txt'
START_SEQUENCE = 10
END_SEQUENCE = 25

# Preprocessing


In [3]:
df = pd.read_sql(
    """
    SELECT 
        timestamp_utc AS timestamp,
        acceleration_x,
        acceleration_y,
        acceleration_z,
        yaw,
        position_x,
        position_y,
        position_z,
        speed 
    FROM telemetry_samples
    WHERE distance_traveled != 0
    ORDER BY id
    """,
    DATABASE_GOOD
)

def preprocess(df):
    df['timestamp'] = pd.to_datetime(df["timestamp"], utc=True).astype("int64") // 10**6
    df['timestamp'] = (df['timestamp'] - min(df['timestamp'])) / 1000

    df['position_x'] = df['position_x'].astype(int)
    df['position_y'] = df['position_y'].astype(int)
    df['position_z'] = df['position_z'].astype(int)

    df['acceleration_x'] = df['acceleration_x'].astype(int)
    df['acceleration_y'] = df['acceleration_y'].astype(int)
    df['acceleration_z'] = df['acceleration_z'].astype(int)
    df['yaw'] = df['yaw'].round(decimals=2)
    df['speed'] = (df['speed'] * 3.6).astype(int)
    
    return df

df_fast = preprocess(df)

# df_fast = df_fast.iloc[::19, :]
# print(df_fast.shape)

In [ ]:
def match_lines_by_euclid(df_slow, df_fast):
    # Convert all columns to object dtype to allow storing tuples
    for col in df_slow.columns:
        df_slow[col] = df_slow[col].astype(object)
    
    #for line in df_slow:
    for _, line in df_slow.iterrows():
        # find line in optimal_df with minimal euclidian in x and z, with distance in y < 2
        optimal_line = find_optimal_line(df_fast, line)

        # write all optimal values into df_slow with the new values being second in a tuple, e.g. "timestamp": 0.0 -> "timestamp": [0.0,0.0]
        for column in df_slow.columns:
            optimal_val = optimal_line[column]
            slow_val = df_slow.at[line.name, column]
            
            # Convert numpy types to Python types
            slow_val = float(slow_val) if isinstance(slow_val, float) else int(slow_val) if isinstance(slow_val, int) else slow_val
            optimal_val = float(optimal_val) if isinstance(optimal_val, float) else int(optimal_val) if isinstance(optimal_val, int) else optimal_val
            
            df_slow.at[line.name, column] = (slow_val, optimal_val)
        # cut off all lines before found line in optimal df without 
        df_fast = df_fast[df_fast.index > optimal_line.name]
    return df_slow


def find_optimal_line(df_fast, line):
    min_distance = sys.maxsize
    optimal_line = None

    for _, fast_line in df_fast.iterrows():
        if abs(fast_line['position_y'] - line['position_y']) < 2:
            distance = ((fast_line['position_x'] - line['position_x']) ** 2 + 
                        (fast_line['position_z'] - line['position_z']) ** 2) ** 0.5
            if distance < min_distance:
                min_distance = distance
                optimal_line = fast_line

    return optimal_line

In [8]:
df2 = pd.read_sql(
    """
    SELECT 
        timestamp_utc AS timestamp,
        acceleration_x,
        acceleration_y,
        acceleration_z,
        yaw,
        position_x,
        position_y,
        position_z,
        speed 
    FROM telemetry_samples
    WHERE distance_traveled != 0
    ORDER BY id
    """,
    DATABASE_BAD
)

df_slow = preprocess(df2)

# x = max(df_slow["timestamp"]) / max(df_fast["timestamp"])
# df_slow = df_slow.iloc[::int(19*x), :]

df_slow = df_slow.iloc[::int(19), :]
print(df_slow.shape)

(72, 9)


In [10]:
# Match df_fast to df_slow
df_slow = match_lines_by_euclid(df_slow, df_fast)

C:\Users\am200\AppData\Local\Temp\ipykernel_9392\3867545886.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_slow[col] = df_slow[col].astype(object)
C:\Users\am200\AppData\Local\Temp\ipykernel_9392\3867545886.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_slow[col] = df_slow[col].astype(object)
C:\Users\am200\AppData\Local\Temp\ipykernel_9392\3867545886.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = va

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [7]:
# df_combined = df_fast.copy()
# for col in df_fast.columns:
#     df_combined[col] = list(zip(df_fast[col], df_slow[col]))

records = df_slow.to_dict(orient="records")
text = str(records)
for char in '[]{}()':
    text = text.replace(char, '')
    
text = text.replace('timestamp', '\ntimestamp')

with open(OUTPUT, "w", encoding="utf-8") as f:
    f.write(text)

In [20]:
with open(OUTPUT) as input_file:
    text = f'{input_file.readlines()[START_SEQUENCE:END_SEQUENCE]}'

text = text.strip('"')
text.replace('\'', '')

system_prompt = "Du bist ein erfahrener Rennfahrer-Coach. Du analysierst Fahrdaten und gibst präzises Feedback zu Linie, Bremspunkten, Gas/Bremse-Dosierung. Antworte kurz, fokussiert, praxisnah, maximal 2 Sätze. Timestamp ist immer in Sekunden. Jedes Attribut hat zwei Werte: Das zweite steht immer für die zu bewertende Runde, der erste Wert beschreibt eine optimale Runde die dir als Referenz dient. **Verwende dafür ausschließlich die Daten aus der Liste des Users**. Negative Prompt: Denk dir keine weiteren Daten aus, Bewerte nicht die ersten Werte der Attribute"
user_prompt = "Bewerte meine Fahrleistung, zeige mir klar die Unterschiede:"

resp = ollama.chat(
    model="nemotron-3-nano:30b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + f"\n```json\n{text}\n```"},
    ],
)

print(resp["message"]["content"])
pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
with open(PROMPTS, "a", encoding="utf-8") as file:
    file.writelines([
        f"Sequence: {START_SEQUENCE} - {END_SEQUENCE}\n",
        "System Prompt: " + str(system_prompt) + "\n",
        "User Prompt: " + str(user_prompt) + "\n",
        "Response: " + str(resp["message"]["content"]) + "\n\n",
    ])

Ab etwa 9 s erreicht du Lateralbeschleunigungen von bis +20 m/s², während die Referenzrunde nur 0‑5 m/s² vorsieht – das erzeugt starkes Unter‑/Übersteuern. Später (ab 21 s) bremst du zu spät, sodass die Geschwindigkeit abrupt von 198 → 96 km/h sinkt und das Fahrzeug bei Yaw‑Werten um –0,5 rad aus der idealen Linie gerät.
